In [0]:
import pyspark.sql.functions  as F
df = (
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("pathGlobFilter", "*.csv")
    .option("header", "true")
    .option(
        "cloudFiles.schemaLocation",
        "/Volumes/citibike/00_landing/citibike_files/checkpoint/autoloader"
    )
    .load("/Volumes/citibike/00_landing/citibike_files/autoloader_input")
    
) # add new columns is default mode


In [0]:
from pyspark.sql.functions import col
(
    df
    .withColumn("__file", col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .writeStream
    .option("checkpointLocation", "/Volumes/citibike/00_landing/citibike_files/checkpoint/autoloader")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("citibike.01_bronze.citibikes_raw")
)

In [0]:
%%sql
SELECT * FROM citibike.01_bronze.citibikes_raw;

In [0]:
%skip
%sql
select __file, count(1)
from citibike.01_bronze.citibikes_raw
group by __file;

In [0]:
# %%sql
# DROP TABLE IF EXISTS citibike.01_bronze.citibikes_raw